In [14]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. CONFIGURATION: MAX FEATURES (35+ Variables)
# ==========================================
VAR_MAP = {
    # --- GEOGRAPHY ---
    'State_FIPS': ['_STATE', 'STATE', 'FIPS'],
    
    # --- DEMOGRAPHICS ---
    'Age': ['_AGE80', '_AGE_G', 'AGE'],
    'Sex': ['SEXVAR', 'SEX', '_SEX'],  
    'Education': ['EDUCA', 'EDUCA'],   
    'Income': ['INCOME3', 'INCOME2', 'INCOME'], 
    'MaritalStatus': ['MARITAL', 'MARITAL'], 
    'Employment': ['EMPLOY1', 'EMPLOY'],     
    'Veteran': ['VETERAN3', 'VETERAN'], 
    
    # --- VITAL SIGNS & HISTORY ---
    'BMI': ['_BMI5', 'BMI'],
    'HighBP': ['_RFHYPE5', '_RFHYPE6', 'BPHIGH4'],  
    'HighChol': ['_RFCHOL', '_RFCHOL1', 'TOLDHI2', 'TOLDHI3'], 
    
    # --- LIFESTYLE RISK FACTORS ---
    'Smoker': ['_RFSMOK3', 'SMOKER3', '_SMOKER3'], 
    'HeavyDrinker': ['_RFDRHV7', '_RFDRHV5', '_RFDRHV8'], 
    'PhysicalActivity': ['_TOTINDA', '_PACAT1'], 
    'FruitCons': ['_FRUTSUM', '_FRUTSU1'], 
    'VegCons': ['_VEGESUM', '_VEGESU1'],   
    
    # --- GENERAL HEALTH & DISABILITY ---
    'GenHealth': ['GENHLTH'], 
    'PhysicalHealthDays': ['PHYSHLTH'], 
    'MentalHealthDays': ['MENTHLTH'],   
    'HealthcareAccess': ['CHECKUP1'],   
    'Blind': ['BLIND', 'CHCVISN1'],      
    'Deaf': ['DEAF', 'CHCHEAR1'],        
    'CognitiveDiff': ['DECIDE'],         
    'DiffWalking': ['DIFFWALK'],         
    'DiffDressing': ['DIFFDRES'],        
    'DiffErrands': ['DIFFALON'],         
    
    # --- DISEASE TARGETS ---
    'HeartAttack': ['CVDINFR4', 'CVDINFR3', 'CVDINFR'],
    'Angina': ['CVDCRHD4', 'CVDCRHD3', 'CVDCRHD'],
    'Stroke': ['CVDSTRK3', 'CVDSTRK'],
    'Asthma': ['ASTHMA3', 'ASTHMA'],
    'SkinCancer': ['CHCSCNCR', 'CHCSCNCR'],
    'OtherCancer': ['CHCOCNCR', 'CHCOCNCR'],
    'COPD': ['CHCCOPD3', 'CHCCOPD2', 'CHCCOPD'],
    'Depression': ['ADDEPEV3', 'ADDEPEV2'],
    'KidneyDisease': ['CHCKDNY2', 'CHCKDNY1'],
    'Diabetes': ['DIABETE4', 'DIABETE3', 'DIABETE2'],
    'Arthritis': ['HAVARTH5', 'HAVARTH4', 'HAVARTH3']
}

FIPS_TO_STATE = {
    1: 'Alabama', 2: 'Alaska', 4: 'Arizona', 5: 'Arkansas', 6: 'California',
    8: 'Colorado', 9: 'Connecticut', 10: 'Delaware', 11: 'District of Columbia',
    12: 'Florida', 13: 'Georgia', 15: 'Hawaii', 16: 'Idaho', 17: 'Illinois',
    18: 'Indiana', 19: 'Iowa', 20: 'Kansas', 21: 'Kentucky', 22: 'Louisiana',
    23: 'Maine', 24: 'Maryland', 25: 'Massachusetts', 26: 'Michigan',
    27: 'Minnesota', 28: 'Mississippi', 29: 'Missouri', 30: 'Montana',
    31: 'Nebraska', 32: 'Nevada', 33: 'New Hampshire', 34: 'New Jersey',
    35: 'New Mexico', 36: 'New York', 37: 'North Carolina', 38: 'North Dakota',
    39: 'Ohio', 40: 'Oklahoma', 41: 'Oregon', 42: 'Pennsylvania',
    44: 'Rhode Island', 45: 'South Carolina', 46: 'South Dakota', 47: 'Tennessee',
    48: 'Texas', 49: 'Utah', 50: 'Vermont', 51: 'Virginia', 53: 'Washington',
    54: 'West Virginia', 55: 'Wisconsin', 56: 'Wyoming'
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_col_name(df, possible_names):
    """Robust column finder."""
    df_cols_upper = [c.upper() for c in df.columns]
    for name in possible_names:
        if name.upper() in df_cols_upper:
            return df.columns[df_cols_upper.index(name.upper())]
    return None

def clean_brfss_year(file_path, year):
    print(f"   -> Loading BRFSS {year}...")
    try:
        df = pd.read_csv(file_path, low_memory=False, encoding='cp1252') 
    except FileNotFoundError:
        print(f"      ERROR: File not found: {file_path}")
        return pd.DataFrame()

    clean_data = pd.DataFrame()
    
    # 1. Extract Columns
    for target_name, possible_cols in VAR_MAP.items():
        raw_col = get_col_name(df, possible_cols)
        if raw_col:
            clean_data[target_name] = df[raw_col]
        else:
            clean_data[target_name] = np.nan

    # 2. Clean Binary Variables
    binary_targets = [
        'HeartAttack', 'Angina', 'Stroke', 'Asthma', 'SkinCancer', 'OtherCancer', 
        'COPD', 'Depression', 'KidneyDisease', 'Diabetes', 'Arthritis', 
        'HighBP', 'HighChol', 'HeavyDrinker', 'PhysicalActivity', 'Smoker', 
        'Sex', 'MaritalStatus', 'Veteran', 'Blind', 'Deaf', 'CognitiveDiff', 
        'DiffWalking', 'DiffDressing', 'DiffErrands'
    ]
    
    for col in binary_targets:
        if col in clean_data.columns:
            # 1=Yes, 2=No. Map 7/9 (Refused) to 0 (No) for Risk Factors.
            clean_data[col] = clean_data[col].replace({2: 0, 7: 0, 9: 0})
            clean_data[col] = clean_data[col].apply(lambda x: 1 if x == 1 else 0)

    # 3. Clean Continuous/Categorical Variables & IMPUTE
    
    # BMI
    if 'BMI' in clean_data.columns:
        clean_data['BMI'] = clean_data['BMI'] / 100
        clean_data['BMI'] = clean_data['BMI'].fillna(clean_data['BMI'].median())
        clean_data = clean_data[(clean_data['BMI'] > 10) & (clean_data['BMI'] < 100)]
        
    # --- SAFE IMPUTATION (Prevents KeyError: 0) ---
    for col in ['Income', 'Education', 'GenHealth', 'Employment']:
        if col in clean_data.columns:
            clean_data[col] = clean_data[col].replace({77: np.nan, 99: np.nan})
            
            # Check if mode exists first!
            mode_series = clean_data[col].mode()
            if not mode_series.empty:
                fill_val = mode_series[0]
            else:
                fill_val = 1 # Default value if column is totally empty
                
            clean_data[col] = clean_data[col].fillna(fill_val)

    # Fruit/Veg
    for col in ['FruitCons', 'VegCons']:
        if col in clean_data.columns:
            clean_data[col] = clean_data[col].fillna(0)

    # Add Metadata
    clean_data['Year'] = year
    if 'State_FIPS' in clean_data.columns:
        clean_data['State'] = clean_data['State_FIPS'].map(FIPS_TO_STATE)
        clean_data.drop(columns=['State_FIPS'], inplace=True)

    # 4. FINAL CLEANUP: Only drop if Disease Target is missing
    disease_targets = ['HeartAttack', 'Angina', 'Stroke', 'Asthma', 'Diabetes']
    clean_data.dropna(subset=[t for t in disease_targets if t in clean_data.columns], inplace=True)

    return clean_data

def clean_epa_year(file_path, year):
    try:
        df = pd.read_csv(file_path)
    except:
        return pd.DataFrame()

    df.columns = [c.lower() for c in df.columns]
    state_col = next((c for c in df.columns if 'state' in c and 'code' not in c), None)
    aqi_col = next((c for c in df.columns if 'aqi' in c or 'mean' in c), None)

    if state_col and aqi_col:
        df[aqi_col] = pd.to_numeric(df[aqi_col], errors='coerce')
        epa_agg = df.groupby(state_col)[aqi_col].mean().reset_index()
        epa_agg.columns = ['State', 'Avg_AQI']
        return epa_agg
    else:
        return pd.DataFrame()

# ==========================================
# 3. EXECUTION (2023 ONLY)
# ==========================================
years = [2023] 
merged_dfs = []

print("--- Starting 2023 MAX-FEATURE Pipeline ---")

for year in years:
    print(f"\nProcessing {year}...")
    
    brfss_file = f'BRFSS{year}.csv'
    brfss_path = os.path.join('data/raw', brfss_file)
    if not os.path.exists(brfss_path):
        brfss_path = os.path.join('data/raw', f'brfss{year}.csv') 
    
    epa_path = os.path.join('data/raw', f'annual_aqi_by_county_{year}.csv')
    
    df_health = clean_brfss_year(brfss_path, year)
    df_epa = clean_epa_year(epa_path, year)
    
    if not df_health.empty:
        if not df_epa.empty:
            df_merged = pd.merge(df_health, df_epa, on='State', how='left')
            df_merged['Avg_AQI'] = df_merged['Avg_AQI'].fillna(df_merged['Avg_AQI'].mean())
        else:
            df_merged = df_health
            df_merged['Avg_AQI'] = 45.0
            
        merged_dfs.append(df_merged)
        print(f"   -> Count of Rows Added: {len(df_merged)}")

if merged_dfs:
    final_df = pd.concat(merged_dfs, ignore_index=True)
    os.makedirs('data/Processed', exist_ok=True)
    final_df.to_csv('data/Processed/multi_year_health_data.csv', index=False)
    print(f"\nSUCCESS: Saved 2023 Dataset with {len(final_df)} rows and {len(final_df.columns)} columns.")
    print("New Columns Include: Blind, Deaf, CognitiveDiff, DiffWalking, Veteran...")

--- Starting 2023 MAX-FEATURE Pipeline ---

Processing 2023...
   -> Loading BRFSS 2023...
   -> Count of Rows Added: 433323

SUCCESS: Saved 2023 Dataset with 433323 rows and 39 columns.
New Columns Include: Blind, Deaf, CognitiveDiff, DiffWalking, Veteran...
